# 02 - Análisis Exploratorio de Datos (EDA) - CFPB

## Consumer Complaint Database

En este notebook se realiza el análisis exploratorio del conjunto de datos **Consumer Complaint Database**, publicado por el Consumer Financial Protection Bureau (CFPB).

El objetivo es estudiar la estructura, calidad y distribución de los datos disponibles, prestando especial atención a las narrativas textuales de los consumidores y a las variables categóricas que podrían utilizarse posteriormente como etiquetas en modelos de clasificación mediante técnicas de Procesamiento del Lenguaje Natural (NLP) y Machine Learning.

Este análisis permitirá determinar la viabilidad del dataset para el desarrollo de un sistema inteligente de clasificación y gestión automática de consultas de clientes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys

print("Versión de Python:", sys.version)
print("Versión de pandas:", pd.__version__)
print("Entorno utilizado:", sys.executable)

Versión de Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Versión de pandas: 3.0.5
Entorno utilizado: c:\Users\Usuario\Documents\UCM Master\TFM Ana Valeria\.venv\Scripts\python.exe


## 1. Carga inicial del dataset

Debido al tamaño del conjunto de datos, se realiza inicialmente una carga parcial de 1.000 registros. Esta muestra permite inspeccionar la estructura del dataset, identificar las variables disponibles y validar el proceso de lectura antes de trabajar con el conjunto completo.

In [2]:
ruta_datos = "../data/raw/complaints.csv"

df_sample = pd.read_csv(
    ruta_datos,
    nrows=1000,
    low_memory=False
)

print("Dimensiones de la muestra:", df_sample.shape)

df_sample.head()

Dimensiones de la muestra: (1000, 16)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-03-28,"Credit reporting, credit repair services, or o...",Other personal consumer report,Incorrect information on your report,Information belongs to someone else,On XX/XX/XXXX I sent in a death certificate an...,NaN,"Scratch Services, Inc.",TX,76706,Servicemember,Web,2023-05-01,Closed with explanation,Yes,6763565
1,2023-11-15,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Old information reappears or never goes away,I have a long time victim of fraud and identit...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NaN,XXXXX,NaN,Web,2023-11-15,Closed with explanation,Yes,7855083
2,2023-11-15,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Old information reappears or never goes away,I have a long time victim of fraud and identit...,NaN,"EQUIFAX, INC.",NaN,XXXXX,NaN,Web,2023-11-15,Closed with explanation,Yes,7856850
3,2023-12-28,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",IL,60411,NaN,Web,2023-12-28,Closed with explanation,Yes,8070678
4,2024-02-12,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,From XXXX XXXX through XXXX XXXX my checking a...,NaN,JPMORGAN CHASE & CO.,TX,782XX,Older American,Web,2024-02-12,Closed with explanation,Yes,8331660


## 2. Inspección de la estructura de los datos

Una vez validada la carga inicial, se analiza la estructura de la muestra para identificar las variables disponibles, sus tipos de datos y la presencia de valores nulos.

Esta primera inspección resulta especialmente relevante para determinar qué variables pueden utilizarse como texto de entrada para los modelos de NLP, cuáles pueden actuar como categorías objetivo y qué información adicional puede resultar útil para el análisis.

In [3]:
print("Número de filas:", df_sample.shape[0])
print("Número de columnas:", df_sample.shape[1])

print("\nColumnas disponibles:")
for i, columna in enumerate(df_sample.columns, start=1):
    print(f"{i}. {columna}")

print("\nTipos de datos:")
print(df_sample.dtypes)

print("\nValores nulos:")
print(df_sample.isnull().sum())

Número de filas: 1000
Número de columnas: 16

Columnas disponibles:
1. Date received
2. Product
3. Sub-product
4. Issue
5. Sub-issue
6. Consumer complaint narrative
7. Company public response
8. Company
9. State
10. ZIP code
11. Tags
12. Submitted via
13. Date sent to company
14. Company response to consumer
15. Timely response?
16. Complaint ID

Tipos de datos:
Date received                     str
Product                           str
Sub-product                       str
Issue                             str
Sub-issue                         str
Consumer complaint narrative      str
Company public response           str
Company                           str
State                             str
ZIP code                          str
Tags                              str
Submitted via                     str
Date sent to company              str
Company response to consumer      str
Timely response?                  str
Complaint ID                    int64
dtype: object

Valores nulo

### Observaciones iniciales

La muestra analizada contiene 16 variables que combinan información textual, categórica, geográfica y temporal.

La variable de mayor interés para el desarrollo de modelos de Procesamiento del Lenguaje Natural es `Consumer complaint narrative`, ya que contiene el texto libre proporcionado por el consumidor. Por su parte, variables como `Product`, `Issue`, `Sub-product` y `Sub-issue` presentan potencial para ser utilizadas como variables objetivo en problemas de clasificación supervisada.

En esta primera muestra se observa una elevada presencia de valores nulos en `Consumer complaint narrative`: 983 de los 1.000 registros no contienen narrativa. Sin embargo, al tratarse únicamente de las primeras filas del archivo, esta proporción no debe considerarse representativa del conjunto completo.

También se observa que determinadas variables, como `Tags` y `Company public response`, presentan una cantidad elevada de valores ausentes. Su utilidad para el proyecto deberá evaluarse posteriormente.

## 3. Disponibilidad de narrativas textuales

Dado que la narrativa del consumidor constituye la principal fuente de información textual para los futuros modelos de NLP, se analiza su disponibilidad en el conjunto completo de datos.

Para reducir el consumo de memoria, en esta fase se carga únicamente la variable `Consumer complaint narrative`.

In [4]:
narrativas = pd.read_csv(
    ruta_datos,
    usecols=["Consumer complaint narrative"],
    low_memory=False
)

total_registros = len(narrativas)
con_narrativa = narrativas["Consumer complaint narrative"].notna().sum()
sin_narrativa = narrativas["Consumer complaint narrative"].isna().sum()

print("Total de registros:", total_registros)
print("Registros con narrativa:", con_narrativa)
print("Registros sin narrativa:", sin_narrativa)

print(
    "Porcentaje con narrativa:",
    round(con_narrativa / total_registros * 100, 2),
    "%"
)

Total de registros: 17021062
Registros con narrativa: 3835352
Registros sin narrativa: 13185710
Porcentaje con narrativa: 22.53 %


### Disponibilidad de información textual

El conjunto completo contiene **17.021.062 registros**, de los cuales **3.835.352 disponen de una narrativa textual del consumidor**. Esto representa aproximadamente el **22,53 % del total de observaciones**.

Aunque una parte importante de los registros no dispone de texto libre, el volumen de narrativas disponibles es suficientemente elevado para desarrollar y evaluar modelos de clasificación de texto mediante técnicas de NLP y Machine Learning.

Además, estos resultados muestran que la muestra inicial formada por las primeras 1.000 observaciones no era representativa respecto a la disponibilidad de narrativas. Por este motivo, las decisiones relativas a la selección de datos se basarán en estadísticas obtenidas sobre el conjunto completo o sobre muestras diseñadas específicamente para el análisis.

In [3]:
columnas_nlp = [
    "Consumer complaint narrative",
    "Product",
    "Issue"
]

df_nlp = pd.read_csv(
    ruta_datos,
    usecols=columnas_nlp,
    low_memory=False
)

# Nos quedamos únicamente con registros que contienen narrativa
df_nlp = df_nlp.dropna(subset=["Consumer complaint narrative"])

print("Dimensiones del dataset NLP:", df_nlp.shape)

print("\nNúmero de productos diferentes:")
print(df_nlp["Product"].nunique())

print("\nNúmero de issues diferentes:")
print(df_nlp["Issue"].nunique())

Dimensiones del dataset NLP: (3835352, 3)

Número de productos diferentes:
21

Número de issues diferentes:
173


## 4. Análisis inicial de las variables objetivo

Tras seleccionar únicamente los registros que contienen una narrativa textual, se obtiene un conjunto de **3.835.352 observaciones**.

Dentro de este subconjunto se identifican **21 categorías diferentes de producto (`Product`)** y **173 categorías diferentes de problema (`Issue`)**.

Estas variables presentan especial interés para el proyecto, ya que pueden utilizarse como etiquetas en problemas de clasificación supervisada. La variable `Product` permitiría desarrollar un primer modelo de clasificación del mensaje según el producto asociado, mientras que `Issue` permitiría abordar posteriormente una clasificación más específica del motivo de la consulta.

Antes de definir definitivamente las variables objetivo, es necesario analizar la distribución de sus categorías y comprobar la existencia de posibles problemas de desbalanceo.

In [6]:
distribucion_productos = (
    df_nlp["Product"]
    .value_counts()
    .to_frame("frecuencia")
)

distribucion_productos["porcentaje"] = (
    distribucion_productos["frecuencia"]
    / len(df_nlp)
    * 100
).round(2)

distribucion_productos

,frecuencia,porcentaje
Product,,
Credit reporting or other personal consumer reports,1671759,43.59
"Credit reporting, credit repair services, or other personal consumer reports",807499,21.05
Debt collection,440528,11.49
Checking or savings account,186416,4.86
Mortgage,146173,3.81
Credit card,126532,3.30
"Money transfer, virtual currency, or money service",120499,3.14
Credit card or prepaid card,108683,2.83
Student loan,62555,1.63


### Distribución de la variable `Product`

La distribución de la variable `Product` presenta un claro desbalanceo entre categorías. Las dos categorías con mayor número de observaciones concentran aproximadamente el 64,6 % de las narrativas disponibles, mientras que algunas categorías presentan una frecuencia muy reducida.

Asimismo, se observan categorías con denominaciones similares que podrían corresponder a modificaciones históricas en la taxonomía utilizada para clasificar los productos. Por ejemplo, existen diferentes categorías relacionadas con *credit reporting*, tarjetas de crédito o préstamos personales.

Este aspecto deberá analizarse antes de construir los modelos de clasificación, ya que mantener categorías históricas muy similares como clases independientes podría introducir complejidad innecesaria y afectar tanto al entrenamiento como a la interpretación de los resultados.

Por tanto, antes de definir la variable objetivo definitiva será necesario estudiar la relación entre estas categorías, su evolución temporal y la posibilidad de establecer una taxonomía más homogénea para el problema de clasificación.

## 5. Evolución temporal de las categorías de producto

Durante el análisis de la variable `Product` se han identificado diferentes categorías con denominaciones similares. Esta situación podría deberse a modificaciones en la taxonomía utilizada por el CFPB a lo largo del tiempo.

Para comprobar esta hipótesis, se analiza la distribución temporal de las categorías de producto. El objetivo es determinar si algunas denominaciones fueron sustituidas por otras en determinados periodos y evaluar posteriormente la conveniencia de homogeneizar estas categorías antes del entrenamiento de los modelos.

In [7]:
df_temporal = pd.read_csv(
    ruta_datos,
    usecols=[
        "Date received",
        "Product",
        "Consumer complaint narrative"
    ],
    low_memory=False
)

# Mantener únicamente registros con narrativa
df_temporal = df_temporal.dropna(
    subset=["Consumer complaint narrative"]
)

# Convertir la fecha a formato datetime
df_temporal["Date received"] = pd.to_datetime(
    df_temporal["Date received"],
    errors="coerce"
)

# Crear variable año
df_temporal["year"] = df_temporal["Date received"].dt.year

print("Dimensiones:", df_temporal.shape)

print("\nRango temporal:")
print("Primer año:", df_temporal["year"].min())
print("Último año:", df_temporal["year"].max())

print("\nFechas no válidas:")
print(df_temporal["Date received"].isna().sum())

Dimensiones: (3835352, 4)

Rango temporal:
Primer año: 2015
Último año: 2026

Fechas no válidas:
0


In [8]:
productos_por_anio = pd.crosstab(
    df_temporal["Product"],
    df_temporal["year"]
)

productos_por_anio

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Product,,,,,,,,,,,,
Bank account or service,4559,7755,2569,0,0,0,0,0,0,0,0,0
Checking or savings account,0,0,4752,6999,7351,9530,12401,18032,28017,29484,51448,18402
Consumer Loan,2969,4666,1823,0,0,0,0,0,0,0,0,0
Credit card,6243,9433,3161,0,0,0,0,0,12905,35451,41517,17822
Credit card or prepaid card,0,0,7799,11854,12526,17102,16607,21512,21283,0,0,0
Credit reporting,10258,15081,6248,0,0,0,0,0,0,0,0,0
Credit reporting or other personal consumer reports,0,0,0,0,0,0,0,0,124058,630670,912757,4274
"Credit reporting, credit repair services, or other personal consumer reports",0,0,36301,48975,58515,95924,109197,233888,224699,0,0,0
Debt collection,14546,18676,23571,26107,23945,25831,34478,31380,35607,70291,100610,35486


### Interpretación de la evolución temporal

El análisis temporal confirma que parte de las categorías de `Product` no representan necesariamente productos completamente diferentes, sino que responden a modificaciones en la taxonomía utilizada por el CFPB a lo largo del tiempo.

Se observa, por ejemplo, la sustitución progresiva de `Bank account or service` por `Checking or savings account`, así como diferentes denominaciones históricas asociadas a *credit reporting*, tarjetas de crédito y determinados tipos de préstamos.

Este resultado es relevante para la construcción del modelo, ya que utilizar directamente las 21 categorías originales podría introducir una separación artificial entre reclamaciones conceptualmente similares únicamente por haber sido registradas en periodos diferentes.

Por este motivo, antes del modelado se evaluará la creación de una taxonomía homogénea que agrupe categorías históricas equivalentes o estrechamente relacionadas.

## 6. Análisis de la variable `Issue`

Además de identificar el producto asociado a cada reclamación, el dataset contiene la variable `Issue`, que representa el motivo o problema principal comunicado por el consumidor.

Esta variable resulta especialmente relevante para los objetivos del proyecto, ya que permite analizar la viabilidad de clasificar automáticamente una consulta no solo según el producto al que hace referencia, sino también según el motivo del contacto.

A continuación, se analiza la distribución de las diferentes categorías de `Issue` para conocer su frecuencia y detectar posibles problemas de desbalanceo.

In [9]:
distribucion_issues = (
    df_nlp["Issue"]
    .value_counts()
    .to_frame("frecuencia")
)

distribucion_issues["porcentaje"] = (
    distribucion_issues["frecuencia"]
    / len(df_nlp)
    * 100
).round(2)

print("Número total de issues:", len(distribucion_issues))

distribucion_issues.head(30)

Número total de issues: 173


,frecuencia,porcentaje
Issue,,
Incorrect information on your report,1204943,31.42
Improper use of your report,664013,17.31
Problem with a company's investigation into an existing problem,349196,9.10
Problem with a credit reporting company's investigation into an existing problem,250448,6.53
Attempts to collect debt not owed,189165,4.93
Managing an account,101559,2.65
Written notification about debt,88542,2.31
Trouble during payment process,56283,1.47
Other transaction problem,54736,1.43


### Interpretación de la distribución de `Issue`

La variable `Issue` presenta una elevada diversidad, con un total de **173 categorías diferentes**, y una distribución claramente desbalanceada.

Las categorías más frecuentes concentran una proporción elevada de las observaciones. En particular, los cuatro `Issue` más frecuentes representan aproximadamente el **64 % de las narrativas disponibles**.

También se observa que varias de las categorías predominantes están relacionadas con problemas de información crediticia (*credit reporting*), lo que resulta coherente con la elevada representación de estos productos observada anteriormente.

Estos resultados indican que utilizar directamente las 173 categorías como variable objetivo supondría un problema de clasificación multiclase considerablemente más complejo que la clasificación por producto. Además, será necesario estudiar la relación existente entre `Product` e `Issue`, ya que determinados problemas pueden estar asociados únicamente o principalmente a determinados tipos de producto.

## 7. Relación entre producto y motivo de la reclamación

Una vez analizadas individualmente las variables `Product` e `Issue`, se estudia la relación existente entre ambas. El objetivo es determinar hasta qué punto los motivos de reclamación están asociados a productos específicos y evaluar la posibilidad de plantear un sistema de clasificación jerárquico.

In [10]:
issues_por_producto = (
    df_nlp
    .groupby("Product")["Issue"]
    .nunique()
    .sort_values(ascending=False)
    .to_frame("numero_issues")
)

issues_por_producto

,numero_issues
Product,
Credit card,45
"Payday loan, title loan, personal loan, or advance loan",33
"Credit reporting, credit repair services, or other personal consumer reports",30
"Payday loan, title loan, or personal loan",26
Credit card or prepaid card,22
Consumer Loan,18
"Money transfer, virtual currency, or money service",17
Mortgage,16
Student loan,14


### Interpretación de la relación entre `Product` e `Issue`

El número de categorías de `Issue` asociadas a cada producto varía considerablemente. Algunos productos, como `Credit card`, presentan hasta 45 motivos de reclamación diferentes, mientras que otros disponen de un conjunto mucho más reducido.

Este resultado evidencia que `Product` e `Issue` mantienen una relación jerárquica: el conjunto de posibles motivos de una reclamación depende, al menos parcialmente, del producto al que pertenece.

Esta estructura resulta especialmente relevante para el objetivo del proyecto, ya que permite plantear un sistema de clasificación en dos niveles. En una primera etapa, el sistema podría identificar automáticamente el producto o área asociada al mensaje y, posteriormente, clasificar el motivo específico de la consulta dentro del conjunto de categorías correspondiente a dicho producto.

### ¿Un mismo `Issue` puede pertenecer a varios productos?

In [11]:
productos_por_issue = (
    df_nlp
    .groupby("Issue")["Product"]
    .nunique()
    .sort_values(ascending=False)
    .to_frame("numero_productos")
)

productos_por_issue.head(30)

,numero_productos
Issue,
Unable to get your credit report or credit score,10
Problem with fraud alerts or security freezes,10
Improper use of your report,10
Incorrect information on your report,10
Credit monitoring or identity theft protection services,10
Problem with a company's investigation into an existing problem,7
Problem with a credit reporting company's investigation into an existing problem,7
Fraud or scam,6
Problem when making payments,5


In [12]:
print("Distribución del número de productos asociados a cada Issue:")
print(productos_por_issue["numero_productos"].value_counts().sort_index())

print(
    "\nIssues exclusivos de un único producto:",
    (productos_por_issue["numero_productos"] == 1).sum()
)

print(
    "Issues presentes en más de un producto:",
    (productos_por_issue["numero_productos"] > 1).sum()
)

Distribución del número de productos asociados a cada Issue:
numero_productos
1     101
2      47
3       9
4       7
5       1
6       1
7       2
10      5
Name: count, dtype: int64

Issues exclusivos de un único producto: 101
Issues presentes en más de un producto: 72


### Análisis de la dependencia entre `Product` e `Issue`

El análisis muestra que **101 de los 173 motivos de reclamación (58,4 %)** aparecen asociados exclusivamente a un único producto, mientras que los **72 restantes (41,6 %)** están presentes en más de una categoría de producto.

Por tanto, aunque existe una relación clara entre ambas variables, `Issue` no constituye una subdivisión estrictamente exclusiva de `Product`. Algunos motivos presentan un carácter transversal y pueden aparecer asociados a diferentes productos.

Este resultado sugiere que la información sobre el producto podría ser útil para mejorar la clasificación del motivo de la reclamación, pero no resulta adecuado asumir una correspondencia jerárquica exclusiva entre ambas variables.

De cara al modelado, se considerará inicialmente la clasificación de `Product` como una tarea independiente y, posteriormente, se evaluará la clasificación de `Issue` como un problema de mayor granularidad.

## 8. Análisis de calidad del conjunto de datos

Antes de realizar el preprocesamiento del texto y construir los modelos de clasificación, se analiza la calidad de los registros seleccionados. En esta etapa se estudia la presencia de valores nulos, registros duplicados y características básicas de las narrativas que puedan afectar al posterior proceso de modelado.

### 8.1 Análisis de valores nulos y vacíos

Una vez seleccionado el subconjunto de registros que contiene narrativa textual, se analiza la presencia de valores nulos o vacíos en las variables que resultan relevantes para el posterior proceso de modelado.

Aunque la selección previa garantiza la existencia de una narrativa, es necesario comprobar también la calidad de las variables `Product` e `Issue`, que podrían utilizarse como variables objetivo en los modelos de clasificación.

In [13]:
columnas_modelado = [
    "Consumer complaint narrative",
    "Product",
    "Issue"
]

for columna in columnas_modelado:
    
    nulos = df_nlp[columna].isna().sum()
    
    vacios = (
        df_nlp[columna]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    
    print(f"{columna}:")
    print(f"  Valores nulos: {nulos}")
    print(f"  Valores vacíos: {vacios}")
    print()

Consumer complaint narrative:
  Valores nulos: 0
  Valores vacíos: 0

Product:
  Valores nulos: 0
  Valores vacíos: 0

Issue:
  Valores nulos: 0
  Valores vacíos: 0



### 8.2 Análisis de registros duplicados

La existencia de textos duplicados puede afectar al entrenamiento y a la evaluación de los modelos, especialmente si una misma narrativa aparece simultáneamente en los conjuntos de entrenamiento y prueba.

Por este motivo, se analiza tanto la existencia de registros completamente duplicados como la repetición de narrativas dentro del conjunto de datos.

In [14]:
duplicados_completos = df_nlp.duplicated().sum()

narrativas_duplicadas = (
    df_nlp["Consumer complaint narrative"]
    .duplicated()
    .sum()
)

print("Registros completamente duplicados:", duplicados_completos)
print("Narrativas duplicadas:", narrativas_duplicadas)

print(
    "Porcentaje de narrativas duplicadas:",
    round(narrativas_duplicadas / len(df_nlp) * 100, 2),
    "%"
)

Registros completamente duplicados: 1241140
Narrativas duplicadas: 1253319
Porcentaje de narrativas duplicadas: 32.68 %


In [15]:
conflictos_product = (
    df_nlp
    .groupby("Consumer complaint narrative")["Product"]
    .nunique()
)

conflictos_issue = (
    df_nlp
    .groupby("Consumer complaint narrative")["Issue"]
    .nunique()
)

print(
    "Narrativas asociadas a más de un Product:",
    (conflictos_product > 1).sum()
)

print(
    "Narrativas asociadas a más de un Issue:",
    (conflictos_issue > 1).sum()
)

Narrativas asociadas a más de un Product: 4436
Narrativas asociadas a más de un Issue: 8556


### Interpretación del análisis de calidad

El subconjunto seleccionado para el análisis NLP no presenta valores nulos ni vacíos en las variables `Consumer complaint narrative`, `Product` e `Issue`, por lo que las variables principales necesarias para el modelado presentan una adecuada completitud.

Sin embargo, se detecta una presencia elevada de duplicados. Existen **1.241.140 registros completamente duplicados** y **1.253.319 ocurrencias adicionales de narrativas ya presentes en el conjunto de datos**, lo que representa un **32,68 % de los registros analizados**.

La elevada presencia de textos repetidos resulta especialmente relevante para el posterior proceso de modelado. Si distintas copias de una misma narrativa se distribuyeran entre los conjuntos de entrenamiento y prueba, podría producirse fuga de información (*data leakage*) y obtenerse una estimación excesivamente optimista del rendimiento del modelo.

Además, se identifican **4.436 narrativas asociadas a más de una categoría de `Product`** y **8.556 asociadas a más de una categoría de `Issue`.** Estos casos requieren un tratamiento específico, ya que pueden representar inconsistencias de etiquetado o situaciones en las que textos idénticos han sido clasificados de manera diferente.

### 8.3 Análisis de narrativas con etiquetas contradictorias

Además de identificar la presencia de textos repetidos, resulta necesario comprobar si una misma narrativa ha sido asociada a diferentes etiquetas.

Este análisis permite distinguir entre duplicados exactos, que representan principalmente redundancia en los datos, y casos potencialmente conflictivos en los que un mismo contenido textual presenta diferentes clasificaciones.

In [16]:
narrativas_conflictivas_product = conflictos_product[
    conflictos_product > 1
].index

ejemplos_conflictos_product = (
    df_nlp[
        df_nlp["Consumer complaint narrative"].isin(
            narrativas_conflictivas_product
        )
    ]
    .sort_values("Consumer complaint narrative")
)

ejemplos_conflictos_product[
    ["Consumer complaint narrative", "Product", "Issue"]
].head(20)

,Consumer complaint narrative,Product,Issue
7249719,""" I am filing this complaint regarding an inac...",Debt collection,False statements or representation
7251120,""" I am filing this complaint regarding an inac...",Credit reporting or other personal consumer re...,Problem with a company's investigation into an...
4835308,""" I am writing to delete the following informa...",Credit reporting or other personal consumer re...,Incorrect information on your report
5466678,""" I am writing to delete the following informa...",Credit reporting or other personal consumer re...,Incorrect information on your report
5466588,""" I am writing to delete the following informa...",Credit reporting or other personal consumer re...,Incorrect information on your report
2054032,""" I am writing to delete the following informa...",Debt collection,Attempts to collect debt not owed
16265138,""" I am writing to delete the following informa...","Credit reporting, credit repair services, or o...",Incorrect information on your report
5465858,""" I am writing to delete the following informa...",Credit reporting or other personal consumer re...,Incorrect information on your report
5509535,""" I am writing to delete the following informa...",Credit reporting or other personal consumer re...,Incorrect information on your report
5509971,""" I am writing to delete the following informa...",Credit reporting or other personal consumer re...,Incorrect information on your report


In [17]:
resumen_conflictos_product = (
    ejemplos_conflictos_product
    .groupby("Consumer complaint narrative")
    .agg(
        numero_productos=("Product", "nunique"),
        productos=("Product", lambda x: list(x.unique())),
        numero_issues=("Issue", "nunique"),
        issues=("Issue", lambda x: list(x.unique()))
    )
    .sort_values("numero_productos", ascending=False)
)

resumen_conflictos_product.head(20)

,numero_productos,productos,numero_issues,issues
Consumer complaint narrative,,,,
"This particular account situation that is lately filing on my own credit document has a seriously unfavorable relation to my personal ability to obtain a present loan application. I highly recommend you generate verification that Credit Union has been reported completely in accordance with the Fair Credit Reporting Act regulations, it's really a serious problem to misreport. More confirmation of the aforesaid item too. My proper request must over, I was never 30 days/60 days/120 days late in any of my payments and I'm not greatly tuned in to the date opened so I prefer to ask you be investigated as soon as possible and confirmed to be correct. Thanks!",7,"[Credit reporting, credit repair services, or ...",3,[Problem with a credit reporting company's inv...
See the attached documents. I want the bureau to start the investigation on these accounts that I am never late for but they're reporting me as late.,6,"[Credit reporting, credit repair services, or ...",4,[Problem with a credit reporting company's inv...
"This particular account situation that is lately filing on my own credit document has a seriously unfavorable relation to my personal ability to obtain a present loan application. I highly recommend you generate verification that Credit Union has been reported completely in accordance with the Fair Credit Reporting Act regulations, it's really a serious problem to misreport. Moreconfirmation of the aforesaid item too. My proper request mustover, I was never 30 days/60 days/120 days late in any of my payments and I'm not greatly tuned in to the date opened so I prefer to ask you be investigated as soon as possible and confirmed to be correct. Thanks!",6,"[Debt collection, Credit card or prepaid card,...",2,"[Attempts to collect debt not owed, Problem wi..."
I see multiple 30 days late marks which is a clear violation of my right under the FCRA. The company has never responded to any of my attempts to obtain any proof or documentation that will prove this account is being reported accurately.,6,"[Credit card or prepaid card, Mortgage, Credit...",2,[Problem with a credit reporting company's inv...
"15 U.S. Code $ 1681c-2 a consumer reporting agency shall block the reporting of any information in the file of a consumer that the consumer identifies as information that resulted from an alleged identity theft, not later than 4 business days after the date of receipt.\n\nIt has been 30 days and you are in VIOLATION of this law because I am a victim of identity theft!! Please delete these items IMMEDIATELY!\n\nThese accounts should not be furnished on my consumer report as they are in VIOLATION!!!\n\nUnder, 15 U.S Code 1681b - Permissible purposes of consumer reports ( a ) IN GENERAL Subject to subsection ( c ) any consumer reporting agency may furnish a consumer report under the following circumstances and no other : ( 2 ) In accordance with the WRITTEN INSTRUCTION of the consumer to whom it relates. I NEVER gave any consumer reporting agency WRITTEN CONSENT to report anything on my consumer report which violates my rights as a federal protected consumer. NO CONSENT IS IDENTITY THEFT. As a consumer I am demanding the deletion of the accounts listed IMMEDIATELY!",6,"[Debt collection, Vehicle loan or lease, Credi...",7,"[Attempts to collect debt not owed, Struggling..."
"In accordance with the Fair Credit Reporting act XXXX Account # XXXX, has violated my rights. \n\n15 U.S.C 1681 section 602 A. States I have the right to privacy.\n\n15 U.S.C 1681 Section 604 A Section 2 : It also states a consumer reporting agency can not furnish a account without my written instructions",6,"[Credit reporting, credit repair services, or ...",8,"[Improper use of your report, Attempts to coll..."
I see multiple 30 & 60-days late marks which is a clear violation of my right under the FCRA. The company has never responded to any of my attempts to obtain any proof or docu

### Interpretación de las etiquetas contradictorias

La inspección de las narrativas asociadas a diferentes etiquetas confirma que los conflictos detectados no responden exclusivamente a cambios históricos en la taxonomía de productos.

Se observan textos idénticos asociados a diferentes categorías de `Product` e `Issue`. En algunos casos, una misma narrativa aparece vinculada a un número elevado de productos y motivos distintos. Este comportamiento puede estar relacionado con la reutilización de textos similares o plantillas por parte de los consumidores, así como con diferencias en el contexto específico de cada reclamación que no necesariamente quedan reflejadas en la narrativa textual.

Este fenómeno resulta especialmente relevante para un problema de clasificación supervisada basado exclusivamente en texto. Si una misma entrada textual está asociada a diferentes etiquetas objetivo, el modelo no dispone de información suficiente para determinar de forma inequívoca cuál de ellas debe predecir.

Por tanto, será necesario definir una estrategia de tratamiento de los textos duplicados y, especialmente, de aquellos que presentan etiquetas contradictorias antes de construir los conjuntos de entrenamiento y evaluación.

### 8.4 Cuantificación de los conflictos de etiquetado

Tras identificar narrativas idénticas asociadas a diferentes categorías, se cuantifica su peso relativo dentro del conjunto de datos. Esta evaluación permitirá determinar si los conflictos representan un problema generalizado o un fenómeno minoritario dentro del corpus.

In [18]:
total_narrativas_unicas = df_nlp["Consumer complaint narrative"].nunique()

conflictos_product_n = (conflictos_product > 1).sum()
conflictos_issue_n = (conflictos_issue > 1).sum()

print("Narrativas únicas:", total_narrativas_unicas)

print("\nConflictos en Product:")
print("Número:", conflictos_product_n)
print(
    "Porcentaje sobre narrativas únicas:",
    round(conflictos_product_n / total_narrativas_unicas * 100, 3),
    "%"
)

print("\nConflictos en Issue:")
print("Número:", conflictos_issue_n)
print(
    "Porcentaje sobre narrativas únicas:",
    round(conflictos_issue_n / total_narrativas_unicas * 100, 3),
    "%"
)

Narrativas únicas: 2582033

Conflictos en Product:
Número: 4436
Porcentaje sobre narrativas únicas: 0.172 %

Conflictos en Issue:
Número: 8556
Porcentaje sobre narrativas únicas: 0.331 %


### Interpretación de los conflictos de etiquetado

Aunque se ha observado una elevada presencia de narrativas repetidas en el conjunto de datos, los casos en los que un mismo texto aparece asociado a etiquetas diferentes representan una proporción reducida del corpus.

De las **2.582.033 narrativas únicas**, únicamente **4.436 (0,172 %)** aparecen asociadas a más de una categoría de `Product`, mientras que **8.556 (0,331 %)** presentan más de una categoría de `Issue`.

Por tanto, el principal problema de calidad identificado no corresponde a una inconsistencia generalizada en las etiquetas, sino a la elevada redundancia derivada de la repetición de narrativas.

De cara al modelado, será conveniente eliminar las observaciones redundantes para evitar que textos idénticos aparezcan simultáneamente en los conjuntos de entrenamiento y prueba. Asimismo, las narrativas que presenten etiquetas contradictorias deberán tratarse de forma específica para evitar introducir ambigüedad en el aprendizaje supervisado.

### 8.5 Análisis de la longitud de las narrativas

La longitud de los textos constituye otro indicador relevante de calidad para el posterior procesamiento mediante técnicas de NLP. Las narrativas excesivamente cortas pueden contener poca información útil para la clasificación, mientras que textos muy extensos pueden incrementar considerablemente el coste computacional.

Por este motivo, se analiza la distribución de la longitud de las narrativas en términos de número de palabras.

In [5]:
longitud_palabras = (
    df_nlp["Consumer complaint narrative"]
    .str.count(r"\S+")
)

print("Estadísticas de longitud (número de palabras):")
print(longitud_palabras.describe())

print("\nNarrativas con menos de 5 palabras:",
      (longitud_palabras < 5).sum())

print("Narrativas con menos de 10 palabras:",
      (longitud_palabras < 10).sum())

print("Narrativas con menos de 20 palabras:",
      (longitud_palabras < 20).sum())

print("Narrativas con más de 500 palabras:",
      (longitud_palabras > 500).sum())

print("Narrativas con más de 1.000 palabras:",
      (longitud_palabras > 1000).sum())

Estadísticas de longitud (número de palabras):
count    3.835352e+06
mean     1.774856e+02
std      2.229186e+02
min      1.000000e+00
25%      6.100000e+01
50%      1.170000e+02
75%      2.140000e+02
max      6.469000e+03
Name: Consumer complaint narrative, dtype: float64

Narrativas con menos de 5 palabras: 3188
Narrativas con menos de 10 palabras: 28836
Narrativas con menos de 20 palabras: 156978
Narrativas con más de 500 palabras: 206266
Narrativas con más de 1.000 palabras: 40523


### Optimización del cálculo

Durante el análisis se realizó inicialmente el cálculo de la longitud de las narrativas mediante la separación de cada texto en una lista de palabras utilizando `str.split().str.len()`.

Sin embargo, al aplicar esta operación sobre más de **3,8 millones de narrativas**, el proceso presentó un elevado coste computacional y de memoria, ya que requería generar estructuras intermedias con las palabras de cada documento.

Por este motivo, se sustituyó esta aproximación por un procedimiento más eficiente basado en `str.count(r"\S+")`, que permite estimar el número de palabras contando directamente las secuencias de caracteres separadas por espacios, sin necesidad de generar una lista de palabras para cada narrativa.

Con esta optimización fue posible realizar el cálculo sobre el corpus completo en un tiempo considerablemente menor y con un uso más eficiente de los recursos disponibles.

Esta decisión pone de manifiesto la importancia de considerar no solo la corrección del procedimiento, sino también su eficiencia y escalabilidad cuando se trabaja con conjuntos de datos de gran volumen.

### Interpretación de la longitud de las narrativas

Las narrativas presentan una longitud media aproximada de **177 palabras** y una mediana de **117 palabras**, lo que indica que, en general, los textos contienen una cantidad de información suficiente para abordar tareas de procesamiento del lenguaje natural y clasificación automática.

La distribución presenta cierta asimetría hacia textos largos, ya que el 75 % de las narrativas contiene hasta **214 palabras**, mientras que la longitud máxima alcanza las **6.469 palabras**.

En los extremos de la distribución se identifican **3.188 narrativas con menos de 5 palabras**, **28.836 con menos de 10 palabras** y **156.978 con menos de 20 palabras**. Por otro lado, **206.266 narrativas superan las 500 palabras** y **40.523 superan las 1.000 palabras**.

Estos resultados muestran la existencia de textos de longitud muy diversa. No obstante, antes de establecer umbrales de exclusión o truncamiento se analizarán las necesidades específicas del modelo y el impacto que estas decisiones puedan tener sobre la información disponible.